# SSL Experiment — nb05: Circuit Analysis

**Goal:** Apply CTLS-specific interpretability tools to the SSL-trained model
and examine whether the circuit scaffold generalizes to novel classes.

**Four analyses:**
1. **Zero-shot circuit silhouette** — cluster quality on CIFAR-100 novel class images
   (no fine-tuning). CTLS-SSL should outperform SimCLR.
2. **Trajectory divergence for novel classes** — at which layer does a novel-class
   image converge to (or diverge from) a known CIFAR-10 circuit cluster?
3. **Neighbor pair visualization** — confirm that mined pairs from Phase 2 training
   are semantically coherent.
4. **Monosemanticity on novel inputs** — CTLS-SSL should continue to show
   structured superposition (low mono, high circuit silhouette) for novel classes.

## 0. Setup

In [ ]:
import sys, os

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO_URL = 'https://github.com/jacobposchl/model-interpretability'
    REPO_DIR = '/content/ctls'

    if not os.path.exists(REPO_DIR):
        os.system(f"git clone {REPO_URL} {REPO_DIR}")
        os.system(f"pip install -r {REPO_DIR}/requirements.txt -q")

    from google.colab import drive
    drive.mount('/content/drive')

    DRIVE_BASE = '/content/drive/MyDrive/ctls'
    os.makedirs(f"{DRIVE_BASE}/data/raw", exist_ok=True)
    os.makedirs(f"{DRIVE_BASE}/experiments", exist_ok=True)

    for link, target in [
        (f"{REPO_DIR}/data/raw",    f"{DRIVE_BASE}/data/raw"),
        (f"{REPO_DIR}/experiments", f"{DRIVE_BASE}/experiments"),
    ]:
        if os.path.islink(link): os.unlink(link)
        os.symlink(target, link)

    ROOT = REPO_DIR
else:
    ROOT = os.path.abspath(os.path.join(os.getcwd(), '../..'))

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
os.chdir(ROOT)
print(f"Working directory: {os.getcwd()} ({'Colab' if IN_COLAB else 'local'})")

In [ ]:
import yaml, torch
import numpy as np
import matplotlib.pyplot as plt

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open('configs/ssl/ctls_ssl.yaml') as f:
    config = yaml.safe_load(f)
with open('configs/ssl/simclr.yaml') as f:
    cfg_simclr = yaml.safe_load(f)

DATA_DIR = config['data']['data_dir']

## 1. Load Models

In [ ]:
from models.soft_mask import SoftMask
from models.backbone import CTLSBackbone
from models.meta_encoder import MetaEncoder

def load_model(config, ckpt_path, device):
    mcfg = config['model']; ecfg = mcfg['meta_encoder']
    sm = SoftMask(init_temperature=mcfg.get('soft_mask', {}).get('init_temperature', 1.0))
    bb = CTLSBackbone(arch=mcfg['arch'], num_classes=mcfg['num_classes'],
                       soft_mask=sm, pretrained=False).to(device)
    me = MetaEncoder(layer_dims=bb.layer_dims,
                     hidden_dim=ecfg.get('hidden_dim', 256),
                     embedding_dim=ecfg.get('embedding_dim', 64),
                     encoder_type=ecfg.get('encoder_type', 'mlp')).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    bb.load_state_dict(ckpt['backbone_state'])
    me.load_state_dict(ckpt['meta_encoder_state'])
    bb.eval(); me.eval()
    return bb, me

backbone_ctls,   meta_ctls   = load_model(config,     'experiments/ssl/ctls_ssl/best.pt', DEVICE)
backbone_simclr, meta_simclr = load_model(cfg_simclr, 'experiments/ssl/simclr/best.pt',  DEVICE)
print('Models loaded.')

## 2. Zero-Shot Circuit Silhouette on Novel Classes

No fine-tuning — just run the frozen SSL-trained backbone on CIFAR-100 novel images
and measure circuit embedding cluster quality.

Prediction: CTLS-SSL > SimCLR on circuit silhouette, because the circuit consistency
pressure creates more discriminative internal representations even for unseen classes.

In [ ]:
from data.ssl import CIFAR100ForTransfer
from data.cifar import get_val_transform
from torch.utils.data import DataLoader
from sklearn.metrics import silhouette_score
import torch.nn.functional as F

novel_dataset = CIFAR100ForTransfer(
    root=DATA_DIR, train=False,
    transform=get_val_transform(), split='novel', download=True,
)
novel_loader = DataLoader(novel_dataset, batch_size=128, shuffle=False, num_workers=4)

def collect_circuit_embeddings(backbone, meta_encoder, loader, device):
    all_z, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            _, traj = backbone(x.to(device))
            z = meta_encoder(traj)
            all_z.append(z.cpu()); all_labels.append(y)
    return torch.cat(all_z).numpy(), torch.cat(all_labels).numpy()

z_ctls,   labels = collect_circuit_embeddings(backbone_ctls,   meta_ctls,   novel_loader, DEVICE)
z_simclr, _      = collect_circuit_embeddings(backbone_simclr, meta_simclr, novel_loader, DEVICE)

sil_ctls   = silhouette_score(z_ctls,   labels, metric='cosine', sample_size=min(2000, len(labels)))
sil_simclr = silhouette_score(z_simclr, labels, metric='cosine', sample_size=min(2000, len(labels)))

print(f"Zero-shot circuit silhouette on CIFAR-100 novel classes:")
print(f"  CTLS-SSL: {sil_ctls:.4f}")
print(f"  SimCLR:   {sil_simclr:.4f}")
print(f"  Delta:    {sil_ctls - sil_simclr:+.4f}")

## 3. Trajectory Divergence for Novel Classes

Using `CircuitAnalyzer.trajectory_divergence_curve()`: for each novel-class image,
compute per-layer cosine distance to the nearest CIFAR-10 circuit centroid.

Prediction: CTLS-SSL novel images converge to a known circuit earlier (lower
distance at earlier layers) than SimCLR, especially for 'close' novel classes.

In [ ]:
from evaluation.circuit_analysis import CircuitAnalyzer
from data.cifar import get_standard_loaders

# Collect CIFAR-10 circuit centroids from CTLS-SSL and SimCLR
dcfg = config['data']
_, val_loader_cifar10 = get_standard_loaders(
    data_dir=dcfg['data_dir'], batch_size=256, num_workers=4, augment=False, download=True,
)

analyzer_ctls   = CircuitAnalyzer(backbone_ctls,   meta_ctls,   val_loader_cifar10, DEVICE)
analyzer_simclr = CircuitAnalyzer(backbone_simclr, meta_simclr, val_loader_cifar10, DEVICE)

print('Collecting CIFAR-10 trajectories for CTLS-SSL...')
trajs_ctls, _, labels_c10_ctls = analyzer_ctls.collect_trajectories(max_samples=5000)
print('Collecting CIFAR-10 trajectories for SimCLR...')
trajs_sim, _, labels_c10_sim = analyzer_simclr.collect_trajectories(max_samples=5000)

In [ ]:
from data.ssl import _CIFAR100_FINE_TO_COARSE

# Select 3 'close' novel classes: pick from small_mammals (superclass 16)
# Fine labels for small_mammals: 36=hamster, 50=mouse, 65=rabbit
CLOSE_NOVEL_FINE_LABELS = [36, 50, 65]   # hamster, mouse, rabbit
CIFAR100_CLASS_NAMES = [
    'apple','aquarium_fish','baby','bear','beaver','bed','bee','beetle',
    'bicycle','bottle','bowl','boy','bridge','bus','butterfly','camel',
    'can','castle','caterpillar','cattle','chair','chimpanzee','clock',
    'cloud','cockroach','couch','crab','crocodile','cup','dinosaur',
    'dolphin','elephant','flatfish','forest','fox','girl','hamster',
    'house','kangaroo','keyboard','lamp','lawn_mower','leopard','lion',
    'lizard','lobster','man','maple_tree','motorcycle','mountain','mouse',
    'mushroom','oak_tree','orange','otter','palm_tree','pear','pickup_truck',
    'pine_tree','plain','plate','poppy','porcupine','possum','rabbit',
    'raccoon','ray','road','rocket','rose','sea','seal','shark','shrew',
    'skunk','skyscraper','snail','snake','spider','squirrel','streetcar',
    'sunflower','sweet_pepper','table','tank','telephone','television',
    'tiger','tractor','train','trout','tulip','turtle','wardrobe','whale',
    'willow_tree','wolf','woman','worm'
]

# Collect novel class images for the selected fine labels
novel_by_class = {}
for fine_label in CLOSE_NOVEL_FINE_LABELS:
    idxs = novel_dataset.class_to_indices.get(fine_label, [])
    if idxs:
        imgs = torch.stack([novel_dataset[i][0] for i in idxs[:50]])
        novel_by_class[fine_label] = imgs

print(f"Collected images for fine labels: {list(novel_by_class.keys())}")

In [ ]:
# Plot divergence curves: per-layer distance to nearest CIFAR-10 centroid
n_layers = len(backbone_ctls.layer_dims)

fig, axes = plt.subplots(1, len(novel_by_class), figsize=(15, 5))
if len(novel_by_class) == 1:
    axes = [axes]

for ax, (fine_label, imgs) in zip(axes, novel_by_class.items()):
    cls_name = CIFAR100_CLASS_NAMES[fine_label]

    # CTLS-SSL divergence
    with torch.no_grad():
        _, traj_ctls = backbone_ctls(imgs[:20].to(DEVICE))
        _, traj_sim  = backbone_simclr(imgs[:20].to(DEVICE))

    layer_cents_ctls   = analyzer_ctls.layer_class_centroids(trajs_ctls, labels_c10_ctls)
    layer_cents_simclr = analyzer_simclr.layer_class_centroids(trajs_sim, labels_c10_sim)

    def min_dist_to_any_centroid(traj, layer_cents):
        n_layers = len(traj)
        B = traj[0].shape[0]
        per_layer_dists = []
        for l in range(n_layers):
            h_l = torch.nn.functional.normalize(traj[l], dim=-1)  # [B, D]
            min_dists = []
            for cls_h in layer_cents[l].values():
                cls_h = torch.nn.functional.normalize(cls_h.unsqueeze(0).to(DEVICE), dim=-1)  # [1, D]
                d = 1.0 - (h_l * cls_h).sum(dim=-1)  # [B] cosine distance
                min_dists.append(d.unsqueeze(1))
            if not min_dists:
                per_layer_dists.append(torch.ones(B) * float('nan'))
                continue
            all_dists = torch.cat(min_dists, dim=1)  # [B, n_cls]
            per_layer_dists.append(all_dists.min(dim=1).values)
        return [d.mean().item() for d in per_layer_dists]

    curve_ctls = min_dist_to_any_centroid(traj_ctls, layer_cents_ctls)
    curve_sim  = min_dist_to_any_centroid(traj_sim,  layer_cents_simclr)

    layers = list(range(1, n_layers + 1))
    ax.plot(layers, curve_sim,  'o--', color='steelblue',   label='SimCLR',   lw=2)
    ax.plot(layers, curve_ctls, 's-',  color='darkorange',  label='CTLS-SSL', lw=2)
    ax.set_title(f'{cls_name}\n(close novel class)')
    ax.set_xlabel('Layer'); ax.set_ylabel('Min cos dist to CIFAR-10 centroid')
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax.set_xticks(layers)

fig.suptitle('Trajectory divergence: novel class images → nearest CIFAR-10 circuit centroid', fontsize=11)
fig.tight_layout()
fig.savefig('experiments/ssl/ctls_ssl/novel_trajectory_divergence.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. UMAP: Novel Classes in Circuit Space

Overlay CIFAR-10 circuit clusters (from training) with CIFAR-100 novel class
embeddings. CTLS-SSL should place novel images near semantically related CIFAR-10
clusters (e.g., small_mammals near dog/cat/deer; vehicles_1 near automobile/truck).

In [ ]:
import umap
from matplotlib.lines import Line2D

CIFAR10_CLASSES = ['airplane','automobile','bird','cat','deer',
                   'dog','frog','horse','ship','truck']

# Collect CIFAR-10 val embeddings
z_c10_ctls, y_c10 = [], []
with torch.no_grad():
    for x, y in val_loader_cifar10:
        _, traj = backbone_ctls(x.to(DEVICE))
        z_c10_ctls.append(meta_ctls(traj).cpu())
        y_c10.append(y)
z_c10_ctls = torch.cat(z_c10_ctls)[:3000].numpy()
y_c10 = torch.cat(y_c10)[:3000].numpy()

# CIFAR-100 novel embeddings
z_novel = z_ctls[:500]
y_novel_fine = labels[:500]

# Fit UMAP on CIFAR-10 data, transform both
reducer = umap.UMAP(metric='cosine', n_neighbors=15, min_dist=0.1, random_state=42)
reducer.fit(z_c10_ctls)
emb_c10   = reducer.transform(z_c10_ctls)
emb_novel = reducer.transform(z_novel)

fig, ax = plt.subplots(figsize=(10, 8))

# CIFAR-10 clusters (background, small dots)
colors10 = plt.cm.tab10(np.linspace(0, 1, 10))
for cls_id in range(10):
    mask = y_c10 == cls_id
    ax.scatter(emb_c10[mask, 0], emb_c10[mask, 1],
               c=[colors10[cls_id]], s=4, alpha=0.3, label=CIFAR10_CLASSES[cls_id])

# Novel class overlays (larger, distinct markers)
novel_fine_labels_present = np.unique(y_novel_fine)
markers = ['*', '^', 'D', 'P', 'X', 'v', '<', '>', 's', 'o',
           'h', '+', 'x', '1', '2', '3', '4', '8', 'p', 'H',
           'd', '|', '_', '.', ',', 'o']
for i, fl in enumerate(novel_fine_labels_present[:10]):
    mask = y_novel_fine == fl
    ax.scatter(emb_novel[mask, 0], emb_novel[mask, 1],
               c='black', s=60, marker=markers[i % len(markers)],
               label=f'Novel: {CIFAR100_CLASS_NAMES[fl]}', zorder=5)

ax.set_title('CTLS-SSL: CIFAR-10 circuit clusters + CIFAR-100 novel class overlays')
ax.legend(fontsize=7, ncol=2, loc='upper right')
fig.tight_layout()
fig.savefig('experiments/ssl/ctls_ssl/umap_novel_overlay.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Linear Probe on Circuit Embeddings for Novel Classes

**Hypothesis test:** If CTLS-SSL circuit embeddings are more linearly separable for novel
classes than SimCLR, the circuit scaffold is genuinely load-bearing — the structure is
real but wasn't being exploited by the cosine-centroid few-shot protocol in nb04.

**Method:** For each episode, fit a logistic regression on the K support circuit embeddings
per class, evaluate on 15 query examples. 600 episodes, 5-way 1-shot and 5-way 5-shot.
Same episode structure as nb04, just replacing cosine-centroid with a linear classifier.

In [ ]:
from sklearn.linear_model import LogisticRegression
from collections import defaultdict


def linear_probe_fewshot(z_all, labels_all, n_way=5, k_shot=1, n_query=15,
                         n_episodes=600, seed=42):
    rng = np.random.RandomState(seed)
    unique_classes = np.unique(labels_all)

    class_to_idx = defaultdict(list)
    for i, l in enumerate(labels_all):
        class_to_idx[l].append(i)

    valid_classes = [c for c in unique_classes
                     if len(class_to_idx[c]) >= k_shot + n_query]

    accs = []
    for _ in range(n_episodes):
        episode_classes = rng.choice(valid_classes, n_way, replace=False)

        support_z, support_y, query_z, query_y = [], [], [], []
        for new_label, cls in enumerate(episode_classes):
            chosen = rng.choice(class_to_idx[cls], k_shot + n_query, replace=False)
            support_z.append(z_all[chosen[:k_shot]])
            support_y.extend([new_label] * k_shot)
            query_z.append(z_all[chosen[k_shot:]])
            query_y.extend([new_label] * n_query)

        clf = LogisticRegression(C=1.0, max_iter=1000, solver='lbfgs',
                                 multi_class='multinomial', random_state=0)
        clf.fit(np.vstack(support_z), np.array(support_y))
        accs.append(clf.score(np.vstack(query_z), np.array(query_y)))

    accs = np.array(accs)
    return accs.mean(), 1.96 * accs.std() / np.sqrt(len(accs))


print('Running 5-way 1-shot linear probe on circuit embeddings...')
lp_ctls_1,   ci_ctls_1   = linear_probe_fewshot(z_ctls,   labels, k_shot=1)
lp_simclr_1, ci_simclr_1 = linear_probe_fewshot(z_simclr, labels, k_shot=1)
print('Running 5-way 5-shot linear probe on circuit embeddings...')
lp_ctls_5,   ci_ctls_5   = linear_probe_fewshot(z_ctls,   labels, k_shot=5)
lp_simclr_5, ci_simclr_5 = linear_probe_fewshot(z_simclr, labels, k_shot=5)

print('\n=== 5-way K-shot Linear Probe on Circuit Embeddings (CIFAR-100 novel) ===')
print(f"{'Model':<12} {'1-shot acc':>12} {'1-shot CI':>10} {'5-shot acc':>12} {'5-shot CI':>10}")
print('-' * 58)
print(f"{'SimCLR':<12} {lp_simclr_1:>12.4f} {ci_simclr_1:>10.4f} {lp_simclr_5:>12.4f} {ci_simclr_5:>10.4f}")
print(f"{'CTLS-SSL':<12} {lp_ctls_1:>12.4f} {ci_ctls_1:>10.4f} {lp_ctls_5:>12.4f} {ci_ctls_5:>10.4f}")
print(f'\n--- Delta: CTLS-SSL - SimCLR ---')
print(f'  1-shot delta: {lp_ctls_1 - lp_simclr_1:+.4f}')
print(f'  5-shot delta: {lp_ctls_5 - lp_simclr_5:+.4f}')

In [ ]:
# Compare linear probe vs cosine-centroid (nb04 results hardcoded for reference)
nb04_simclr = {1: 0.4344, 5: 0.5907}
nb04_ctls   = {1: 0.4408, 5: 0.5888}
nb04_ci_sim  = {1: 0.0070, 5: 0.0069}
nb04_ci_ctls = {1: 0.0069, 5: 0.0066}

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
methods = ['Cosine-centroid\n(nb04)', 'Linear probe\n(nb05)']

for ax, k_shot in zip(axes, [1, 5]):
    sim_vals  = [nb04_simclr[k_shot], lp_simclr_1 if k_shot == 1 else lp_simclr_5]
    ctls_vals = [nb04_ctls[k_shot],   lp_ctls_1   if k_shot == 1 else lp_ctls_5]
    sim_cis   = [nb04_ci_sim[k_shot],  ci_simclr_1 if k_shot == 1 else ci_simclr_5]
    ctls_cis  = [nb04_ci_ctls[k_shot], ci_ctls_1   if k_shot == 1 else ci_ctls_5]

    x = np.arange(2)
    width = 0.3
    ax.bar(x - width/2, sim_vals,  width, yerr=sim_cis,  capsize=4,
           label='SimCLR',   color='steelblue',  alpha=0.8)
    ax.bar(x + width/2, ctls_vals, width, yerr=ctls_cis, capsize=4,
           label='CTLS-SSL', color='darkorange', alpha=0.8)

    for i, (s, c, sc, cc) in enumerate(zip(sim_vals, ctls_vals, sim_cis, ctls_cis)):
        ax.text(i, max(s, c) + max(sc, cc) + 0.01,
                f'{c - s:+.3f}', ha='center', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(methods)
    ax.set_ylabel('5-way accuracy')
    ax.set_title(f'5-way {k_shot}-shot')
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(True, axis='y', alpha=0.3)

fig.suptitle('Cosine-centroid vs linear probe: does circuit structure help with a better classifier?',
             fontsize=10)
fig.tight_layout()
os.makedirs('experiments/ssl/ctls_ssl', exist_ok=True)
fig.savefig('experiments/ssl/ctls_ssl/linear_probe_vs_cosine.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Raw Trajectory Linear Probe

**Motivation:** The MetaEncoder compresses the 8-layer backbone trajectory into a
fixed-dim embedding `z`. Sections 2 and 5 show that `z` is no better than SimCLR's
for novel-class transfer, even though the raw trajectory is clearly more circuit-structured
(section 3). If the bottleneck is the MetaEncoder compression, bypassing it — using the
raw trajectory activations directly as features — should reveal whether the circuit
structure is actually discriminative.

**Method:** Concatenate all 8 per-layer GAP activations into a single L2-normalised
feature vector. Run the same 5-way 1-shot / 5-shot linear probe as section 5.
No MetaEncoder involved.

In [ ]:
import torch.nn.functional as F

def collect_traj_embeddings(backbone, loader, device):
    """Concatenate all backbone trajectory layers into one L2-normalised feature."""
    all_feats, all_labels = [], []
    with torch.no_grad():
        for x, y in loader:
            _, traj = backbone(x.to(device))
            # traj: list of [B, D_l] — concat across layers then L2-normalise
            feat = F.normalize(torch.cat(traj, dim=-1), dim=-1).cpu()
            all_feats.append(feat)
            all_labels.append(y)
    return torch.cat(all_feats).numpy(), torch.cat(all_labels).numpy()

print('Collecting raw trajectory embeddings for novel classes...')
traj_ctls,   traj_labels = collect_traj_embeddings(backbone_ctls,   novel_loader, DEVICE)
traj_simclr, _           = collect_traj_embeddings(backbone_simclr, novel_loader, DEVICE)
print(f'Trajectory feature dim — CTLS-SSL: {traj_ctls.shape[1]}, SimCLR: {traj_simclr.shape[1]}')

In [ ]:
print('Running 5-way 1-shot linear probe on raw trajectories...')
lp_traj_ctls_1,   ci_traj_ctls_1   = linear_probe_fewshot(traj_ctls,   traj_labels, k_shot=1)
lp_traj_simclr_1, ci_traj_simclr_1 = linear_probe_fewshot(traj_simclr, traj_labels, k_shot=1)
print('Running 5-way 5-shot linear probe on raw trajectories...')
lp_traj_ctls_5,   ci_traj_ctls_5   = linear_probe_fewshot(traj_ctls,   traj_labels, k_shot=5)
lp_traj_simclr_5, ci_traj_simclr_5 = linear_probe_fewshot(traj_simclr, traj_labels, k_shot=5)

print('\n=== Raw Trajectory Linear Probe (CIFAR-100 novel) ===')
print(f"{'Model':<12} {'1-shot acc':>12} {'1-shot CI':>10} {'5-shot acc':>12} {'5-shot CI':>10}")
print('-' * 58)
print(f"{'SimCLR':<12} {lp_traj_simclr_1:>12.4f} {ci_traj_simclr_1:>10.4f} {lp_traj_simclr_5:>12.4f} {ci_traj_simclr_5:>10.4f}")
print(f"{'CTLS-SSL':<12} {lp_traj_ctls_1:>12.4f} {ci_traj_ctls_1:>10.4f} {lp_traj_ctls_5:>12.4f} {ci_traj_ctls_5:>10.4f}")
print(f'\n--- Delta: CTLS-SSL - SimCLR ---')
print(f'  1-shot delta: {lp_traj_ctls_1 - lp_traj_simclr_1:+.4f}')
print(f'  5-shot delta: {lp_traj_ctls_5 - lp_traj_simclr_5:+.4f}')

In [ ]:
# Three-way comparison: cosine-centroid / MetaEncoder z / raw trajectory
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
methods = ['Cosine-centroid\n(nb04)', 'MetaEncoder z\n(sec 5)', 'Raw trajectory\n(sec 6)']

nb04_simclr  = {1: 0.4344, 5: 0.5907}
nb04_ctls    = {1: 0.4408, 5: 0.5888}
nb04_ci_sim  = {1: 0.0070, 5: 0.0069}
nb04_ci_ctls = {1: 0.0069, 5: 0.0066}

for ax, k_shot in zip(axes, [1, 5]):
    if k_shot == 1:
        sim_vals  = [nb04_simclr[1],  lp_simclr_1,  lp_traj_simclr_1]
        ctls_vals = [nb04_ctls[1],    lp_ctls_1,    lp_traj_ctls_1]
        sim_cis   = [nb04_ci_sim[1],  ci_simclr_1,  ci_traj_simclr_1]
        ctls_cis  = [nb04_ci_ctls[1], ci_ctls_1,    ci_traj_ctls_1]
    else:
        sim_vals  = [nb04_simclr[5],  lp_simclr_5,  lp_traj_simclr_5]
        ctls_vals = [nb04_ctls[5],    lp_ctls_5,    lp_traj_ctls_5]
        sim_cis   = [nb04_ci_sim[5],  ci_simclr_5,  ci_traj_simclr_5]
        ctls_cis  = [nb04_ci_ctls[5], ci_ctls_5,    ci_traj_ctls_5]

    x = np.arange(3)
    width = 0.3
    ax.bar(x - width/2, sim_vals,  width, yerr=sim_cis,  capsize=4,
           label='SimCLR',   color='steelblue',  alpha=0.8)
    ax.bar(x + width/2, ctls_vals, width, yerr=ctls_cis, capsize=4,
           label='CTLS-SSL', color='darkorange', alpha=0.8)

    for i, (s, c, sc, cc) in enumerate(zip(sim_vals, ctls_vals, sim_cis, ctls_cis)):
        ax.text(i, max(s, c) + max(sc, cc) + 0.01,
                f'{c - s:+.3f}', ha='center', fontsize=9)

    ax.set_xticks(x)
    ax.set_xticklabels(methods, fontsize=9)
    ax.set_ylabel('5-way accuracy')
    ax.set_title(f'5-way {k_shot}-shot')
    ax.legend()
    ax.set_ylim(0, 1.0)
    ax.grid(True, axis='y', alpha=0.3)

fig.suptitle('Where is the CTLS-SSL advantage (if any)? Cosine-centroid vs MetaEncoder vs raw trajectory',
             fontsize=10)
fig.tight_layout()
fig.savefig('experiments/ssl/ctls_ssl/three_way_probe_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

| Analysis | CTLS-SSL | SimCLR | Interpretation |
|----------|---------|--------|----------------|
| Zero-shot circuit silhouette (novel classes) | ___ | ___ | Higher = better circuit generalization |
| Divergence curve (close novel, min dist at layer 8) | ___ | ___ | Lower = closer to known circuit |
| UMAP: novel classes near related CIFAR-10 clusters | visual | visual | Validates circuit scaffold |
| Linear probe 1-shot (circuit embeddings) | ___ | ___ | Higher delta = circuit space is load-bearing |
| Linear probe 5-shot (circuit embeddings) | ___ | ___ | Higher delta = circuit space is load-bearing |

**Key question:** Does the linear probe delta (CTLS-SSL - SimCLR) exceed the cosine-centroid
delta from nb04? If yes: the circuit scaffold is real and exploitable; nb04 just used the wrong
evaluation protocol. If no: the circuit structure visible in trajectory divergence is
epiphenomenal — CTLS-SSL routes differently internally but the final embeddings are equivalent.

**Next:** `nb06` — or revisit training if the linear probe still shows no gap.